In [1]:
import gc
import os
from collections import Counter

import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

from dataset.pac_matrix import PAC5MatrixHoFDataset
from gcnn.base_model import GroupEquivariantCNN

K_FOLDS = 5
BATCH_SIZE = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class GCNNDirectPrediction(nn.Module):
    def __init__(self, kernel_size=5, num_hidden=5, hidden_channels=48):
        super().__init__()

        self.feature_extractor = GroupEquivariantCNN(
            property_types=["extensive"],
            kernel_size=kernel_size,
            num_hidden=num_hidden,
            hidden_channels=hidden_channels,
        )

        self.feature_extractor.thermodynamic_head = nn.Identity()
        self.feature_extractor.electronic_head = nn.Identity()

        self.prediction_head = nn.Sequential(
            nn.BatchNorm1d(hidden_channels),
            nn.Linear(hidden_channels, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, pac_tensor):
        _, extensive_features = self.feature_extractor.extract_features(pac_tensor)
        pred_hof = self.prediction_head(extensive_features)

        return pred_hof


dataset = PAC5MatrixHoFDataset(target_hof_col="nist_hof")

stratify_labels = [f"{item[4]}_{item[5]}" for item in dataset.paired_data]
label_counts = Counter(stratify_labels)

safe_stratify_labels = np.array(
    [
        label if label_counts[label] >= K_FOLDS else "rare_combination"
        for label in stratify_labels
    ]
)
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

Using device: cuda
Fetching paired HOF data...
Matched 621 paired samples.


In [2]:
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), safe_stratify_labels), 1
):
    print(f"\n======[ Fold {fold}/{K_FOLDS} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=safe_stratify_labels[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    log_dir = f"../logs/gcnn_3.7M-pac_grid-[PAC5-hof-to_nist_hof]-direct/fold_{fold}"
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir=log_dir)

    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    test_sub = Subset(dataset, test_idx)

    train_loader = DataLoader(
        train_sub, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True
    )
    val_loader = DataLoader(
        val_sub, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
    )
    test_loader = DataLoader(
        test_sub, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
    )

    model = GCNNDirectPrediction().to(device)

    criterion = nn.MSELoss()

    optimizer = optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=5e-3, steps_per_epoch=len(train_loader), epochs=100
    )

    save_dir = "../models/gcnn_3.7M-pac_grid-[PAC5-hof-to_nist_hof]-direct"
    os.makedirs(save_dir, exist_ok=True)
    save_path = f"{save_dir}/fold_{fold}.pth"

    best_val_loss = float("inf")

    for epoch in range(1, 101):
        model.train()
        running_loss = 0.0

        for pac_tensor, _, high_hof, _, _ in train_loader:
            pac_tensor = pac_tensor.to(device)
            high_hof = high_hof.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                pred_hof = model(pac_tensor)
                loss = criterion(pred_hof, high_hof)

            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        model.eval()
        avg_val_loss = 0.0

        with torch.no_grad():
            for pac_tensor, _, high_hof, _, _ in val_loader:
                pac_tensor = pac_tensor.to(device)
                high_hof = high_hof.to(device)

                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.bfloat16,
                    enabled=device.type == "cuda",
                ):
                    pred_hof = model(pac_tensor)
                    avg_val_loss += criterion(pred_hof, high_hof).item()

        avg_val_loss /= len(val_loader)

        lr = optimizer.param_groups[0]["lr"]

        writer.add_scalar("loss/train", avg_train_loss, epoch)
        writer.add_scalar("loss/val", avg_val_loss, epoch)
        writer.add_scalar("hyperparameters/lr", lr, epoch)
        writer.flush()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            marker = " (best)"
            torch.save(model.state_dict(), save_path)
        else:
            marker = ""

        if epoch % 10 == 0 or epoch == 1 or marker:
            print(
                f"Epoch {epoch:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}{marker}"
            )

    print(f"Best Validation Loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    writer.close()

    del train_sub, val_sub, test_sub
    del train_loader, val_loader, test_loader
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Loss per fold: {fold_results}")
print(f"Average Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======
Epoch 001 | Train Loss: 216390.3203 | Val Loss: 223840.5781 (best)
Epoch 002 | Train Loss: 217883.1133 | Val Loss: 223660.9844 (best)
Epoch 003 | Train Loss: 218147.7031 | Val Loss: 223531.2656 (best)
Epoch 004 | Train Loss: 216820.4883 | Val Loss: 223223.7031 (best)
Epoch 005 | Train Loss: 217791.1250 | Val Loss: 223162.7188 (best)
Epoch 006 | Train Loss: 215232.5742 | Val Loss: 222999.6094 (best)
Epoch 010 | Train Loss: 216999.3555 | Val Loss: 216573.7188 (best)
Epoch 011 | Train Loss: 208610.7188 | Val Loss: 195299.6094 (best)
Epoch 013 | Train Loss: 191629.3828 | Val Loss: 179957.8594 (best)
Epoch 014 | Train Loss: 169762.7617 | Val Loss: 154558.0469 (best)
Epoch 017 | Train Loss: 36403.4644 | Val Loss: 27622.3906 (best)
Epoch 019 | Train Loss: 16977.0176 | Val Loss: 9721.5801 (best)
Epoch 020 | Train Loss: 10498.2004 | Val Loss: 29495.0234
Epoch 030 | Train Loss: 4435.8685 | Val Loss: 4254.0640 (best)
Epoch 033 | Train Loss: 8327.2006 | Val Loss: 3506.958

In [3]:
import os
from collections import Counter

import numpy as np
import torch
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset


def evaluate_model(model, dataloader, device):
    model.eval()
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for pac_tensor, _, high_level, _, _ in dataloader:
            pac_tensor = pac_tensor.to(device)

            pred = model(pac_tensor)

            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(high_level.numpy())

    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()

    errors = all_preds - all_targets
    abs_errors = np.abs(errors)

    mae = np.mean(abs_errors)
    rmsd = np.sqrt(np.mean(errors**2))
    max_err = np.max(abs_errors)
    min_err = np.min(abs_errors)
    r2 = r2_score(all_targets, all_preds)

    return mae, rmsd, max_err, min_err, r2


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating on device: {device}")

dataset = PAC5MatrixHoFDataset(target_hof_col="nist_hof")

stratify_labels = [f"{item[4]}_{item[5]}" for item in dataset.paired_data]
k_folds = 5
label_counts = Counter(stratify_labels)

safe_stratify_labels = np.array(
    [
        label if label_counts[label] >= k_folds else "rare_combination"
        for label in stratify_labels
    ]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_train_metrics = []
fold_val_metrics = []
fold_test_metrics = []

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), safe_stratify_labels), 1
):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=safe_stratify_labels[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    test_sub = Subset(dataset, test_idx)

    train_loader = DataLoader(train_sub, batch_size=256, shuffle=False)
    val_loader = DataLoader(val_sub, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_sub, batch_size=256, shuffle=False)

    model = GCNNDirectPrediction().to(device)

    load_path = (
        f"../models/gcnn_3.7M-pac_grid-[PAC5-hof-to_nist_hof]-direct/fold_{fold}.pth"
    )
    try:
        model.load_state_dict(torch.load(load_path, map_location=device))
    except FileNotFoundError:
        print(f"Could not find {load_path}. Skipping this fold.")
        continue

    train_results = evaluate_model(model, train_loader, device)
    val_results = evaluate_model(model, val_loader, device)
    test_results = evaluate_model(model, test_loader, device)

    fold_train_metrics.append(train_results)
    fold_val_metrics.append(val_results)
    fold_test_metrics.append(test_results)

    t_mae, t_rmsd, t_max, t_min, t_r2 = train_results
    print(
        f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
    )

    v_mae, v_rmsd, v_max, v_min, v_r2 = val_results
    print(
        f" + Val -> MAE: {v_mae:.4f}\tRMSD: {v_rmsd:.4f}\tMax: {v_max:.4f}\tMin: {v_min:.4f}\tR^2: {v_r2:.4f}"
    )

    te_mae, te_rmsd, te_max, te_min, te_r2 = test_results
    print(
        f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
    )

if fold_train_metrics and fold_val_metrics and fold_test_metrics:
    print("\n======[ Final 5-fold cross validation results ]======")

    train_metrics_array = np.array(fold_train_metrics)
    val_metrics_array = np.array(fold_val_metrics)
    test_metrics_array = np.array(fold_test_metrics)

    avg_train = np.mean(train_metrics_array, axis=0)
    std_train = np.std(train_metrics_array, axis=0, ddof=1)

    avg_val = np.mean(val_metrics_array, axis=0)
    std_val = np.std(val_metrics_array, axis=0, ddof=1)

    avg_test = np.mean(test_metrics_array, axis=0)
    std_test = np.std(test_metrics_array, axis=0, ddof=1)

    print("Train:")
    print(f" + MAE: {avg_train[0]:.4f} ± {std_train[0]:.4f}")
    print(f" + RMSD: {avg_train[1]:.4f} ± {std_train[1]:.4f}")
    print(f" + Max: {avg_train[2]:.4f} ± {std_train[2]:.4f}")
    print(f" + Min: {avg_train[3]:.4f} ± {std_train[3]:.4f}")
    print(f" + R^2: {avg_train[4]:.4f} ± {std_train[4]:.4f}\n")

    print("Validation:")
    print(f" + MAE: {avg_val[0]:.4f} ± {std_val[0]:.4f}")
    print(f" + RMSD: {avg_val[1]:.4f} ± {std_val[1]:.4f}")
    print(f" + Max: {avg_val[2]:.4f} ± {std_val[2]:.4f}")
    print(f" + Min: {avg_val[3]:.4f} ± {std_val[3]:.4f}")
    print(f" + R^2: {avg_val[4]:.4f} ± {std_val[4]:.4f}\n")

    print("Test:")
    print(f" + MAE: {avg_test[0]:.4f} ± {std_test[0]:.4f}")
    print(f" + RMSD: {avg_test[1]:.4f} ± {std_test[1]:.4f}")
    print(f" + Max: {avg_test[2]:.4f} ± {std_test[2]:.4f}")
    print(f" + Min: {avg_test[3]:.4f} ± {std_test[3]:.4f}")
    print(f" + R^2: {avg_test[4]:.4f} ± {std_test[4]:.4f}")

Evaluating on device: cuda
Fetching paired HOF data...
Matched 621 paired samples.

======[ Evaluating fold 1/5 ]======
 + Train -> MAE: 13.5047	RMSD: 21.1743	Max: 123.9666	Min: 0.0666	R^2: 0.9412
 + Val -> MAE: 15.7537	RMSD: 22.7223	Max: 95.0728	Min: 0.5971	R^2: 0.9338
 + Test -> MAE: 15.5450	RMSD: 27.3882	Max: 148.9748	Min: 0.2423	R^2: 0.9068

======[ Evaluating fold 2/5 ]======
 + Train -> MAE: 8.3769	RMSD: 12.7350	Max: 95.8536	Min: 0.0288	R^2: 0.9786
 + Val -> MAE: 11.6128	RMSD: 17.6501	Max: 57.7658	Min: 0.5072	R^2: 0.9571
 + Test -> MAE: 9.0746	RMSD: 12.9251	Max: 61.2184	Min: 0.0032	R^2: 0.9804

======[ Evaluating fold 3/5 ]======
 + Train -> MAE: 11.7566	RMSD: 16.0516	Max: 70.3090	Min: 0.0733	R^2: 0.9671
 + Val -> MAE: 12.2123	RMSD: 16.8799	Max: 61.4658	Min: 0.3745	R^2: 0.9593
 + Test -> MAE: 15.0434	RMSD: 26.8745	Max: 169.8849	Min: 0.0879	R^2: 0.9064

======[ Evaluating fold 4/5 ]======
 + Train -> MAE: 16.3770	RMSD: 22.8699	Max: 108.1839	Min: 0.0283	R^2: 0.9352
 + Val -> MAE: 2